# LungAI - procesamiento por lotes en Colab

Procesa la muestra del NLST **sin guardar las CT**: por cada lote descarga las series, las convierte, detecta nodulos, recorta cada candidato y borra las CT. En tu Google Drive solo quedan los resultados (JSON, recortes y registro de avance).

- Usa un entorno con **GPU**: menu *Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > T4 GPU*.
- Se puede correr en varias sesiones: cada una sigue donde se quedo la anterior.
- Es un proyecto academico. Detecta nodulos y prepara datos para entrenar; **no es un diagnostico**.

## 1. Revisar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Conectar Google Drive y elegir que procesar

- `SALIDA`: carpeta de Drive donde quedan los resultados.
- `SERIES_POR_SESION`: cuantas series procesar en esta sesion. Empieza con 20 para medir el tiempo.
- `SPLIT`: dejalo vacio para todo, o usa `train`, `validation` o `test`. Sirve para repartir el trabajo entre varias cuentas.
- `WORKERS`: series que se convierten a la vez. Si la sesion se queda sin RAM, baja a 1.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

SALIDA = "/content/drive/MyDrive/lungai"
SERIES_POR_SESION = 20
SPLIT = ""
WORKERS = 2

COHORTE = f"{SALIDA}/monai_dataset/cohorte_cancer_12m.parquet"
REPO = "/content/proyecto-big-data"

## 3. Descargar el proyecto e instalar el entorno

Instala las mismas versiones que el repo (`uv.lock`). La primera vez de cada sesion tarda unos minutos.

In [ ]:
%%bash
cd /content
if [ -d proyecto-big-data ]; then
  git -C proyecto-big-data pull -q
else
  git clone -q https://github.com/EMIL1OCR9/proyecto-big-data.git
fi
pip install -q uv
cd proyecto-big-data
uv sync -q
MPLBACKEND=Agg uv run python -c "import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU')"

## 4. Armar la cohorte y la division (solo la primera vez)

Usa solo las tablas publicas de IDC; no descarga imagenes. La tabla se guarda en Drive para que todas las sesiones usen la misma division de train, validation y test.

In [ ]:
import os
import shutil

if os.path.exists(COHORTE):
    print("Ya existe la cohorte en Drive:", COHORTE)
else:
    !cd {REPO} && MPLBACKEND=Agg uv run python backend/data_pipeline/build_cancer_cohort.py
    !cd {REPO} && MPLBACKEND=Agg uv run python backend/data_pipeline/prepare_monai_dataset.py
    os.makedirs(SALIDA, exist_ok=True)
    shutil.copytree(f"{REPO}/outputs/monai_dataset", f"{SALIDA}/monai_dataset")
    print("Cohorte guardada en Drive:", COHORTE)

## 5. Procesar un lote de series

Las CT se guardan de forma temporal en `/content/ct` y se borran al terminar cada serie. Si la sesion se corta, vuelve a correr desde la celda 2: sigue donde se quedo.

In [ ]:
split = f"--split {SPLIT}" if SPLIT else ""
!cd {REPO} && MPLBACKEND=Agg uv run python backend/inference/procesar_lotes.py --input "{COHORTE}" --data-dir /content/ct --output-dir "{SALIDA}" --model-dir /content/modelos --limit {SERIES_POR_SESION} --workers {WORKERS} {split} 2>&1 | grep --line-buffered -v "Unsupported JPEG"

## 6. Ver el avance

In [ ]:
import os
import pandas as pd

if not os.path.exists(f"{SALIDA}/lotes_progreso.csv"):
    raise SystemExit("Todavia no hay series procesadas: revisa el mensaje de la celda 5.")

avance = pd.read_csv(f"{SALIDA}/lotes_progreso.csv")
muestra = pd.read_parquet(COHORTE)
muestra = muestra[muestra["en_muestra"]]

print(f"Series procesadas: {len(avance)} de {len(muestra)}")
print(f"Descarga procesada: {avance['series_size_MB'].sum() / 1024:.1f} de {muestra['series_size_MB'].sum() / 1024:.1f} GB")
print(f"Nodulos: {int(avance['nodulos'].sum())} | recortes: {int(avance['recortes'].sum())}")
print(f"Deteccion: {avance['segundos_deteccion'].mean():.0f} s por serie en promedio")
print(avance.groupby("split").size().rename("series").to_string())

## 7. Etiquetar los candidatos

Cruza los resultados con las tablas del NLST y marca cada candidato como maligno, benigno o sin etiqueta. Se puede correr en cualquier momento; usa todo lo procesado hasta ahora.

In [ ]:
!cd {REPO} && MPLBACKEND=Agg uv run python backend/inference/etiquetar_nodulos.py --nodules-dir "{SALIDA}/nodulos" --cohort "{COHORTE}" --output-dir "{SALIDA}/etiquetas"

## Que queda en Drive

- `nodulos/`: un JSON por serie con los candidatos del detector.
- `recortes/`: un archivo por serie con el cubo de cada candidato (entrada del clasificador).
- `lotes_progreso.csv`: una fila por serie procesada; `lotes_fallidas.csv`: las que fallaron en la ultima corrida.
- `etiquetas/`: la tabla de candidatos etiquetados y si el detector encontro cada nodulo del radiologo.
- `monai_dataset/`: la cohorte con la division usada.

No compartas la carpeta de forma publica: tiene identificadores de investigacion.